# StockHistory EDA from Parquet

Notebook นี้อ่านข้อมูลจาก Parquet เท่านั้น เพื่อหลีกเลี่ยงการ scan ไฟล์ CSV จำนวนมากซ้ำ ๆ ระหว่างทำ EDA

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jobs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jobs.common import DEFAULT_PARQUET_OUTPUT, DEFAULT_RAW_INPUT_GROUP_BYTES, create_spark

parquet_path = Path(os.getenv("PARQUET_OUTPUT_PATH", str(DEFAULT_PARQUET_OUTPUT))).resolve()
parquet_files = list(parquet_path.rglob("*.parquet")) if parquet_path.exists() else []
assert parquet_files, f"No Parquet files found at {parquet_path}. Run jobs/csv_to_parquet.py first."

spark = create_spark("stockhistory_eda")
df = spark.read.parquet(str(parquet_path)).cache()
print(f"Parquet path: {parquet_path}")
print(f"Parquet files: {len(parquet_files):,}")
df.printSchema()

## Dataset size and 512MB grouping estimate

In [ ]:
from pyspark.sql import functions as F

stockhistory_dir = PROJECT_ROOT / "Data" / "StockHistory"
csv_files = list(stockhistory_dir.glob("*.csv")) if stockhistory_dir.exists() else []
csv_bytes = sum(path.stat().st_size for path in csv_files)
estimated_groups = (csv_bytes + DEFAULT_RAW_INPUT_GROUP_BYTES - 1) // DEFAULT_RAW_INPUT_GROUP_BYTES if csv_bytes else 0

print(f"Source CSV files: {len(csv_files):,}")
print(f"Source CSV size: {csv_bytes / 1024 / 1024:,.2f} MB")
print(f"Estimated 512MB groups: {estimated_groups:,}")
print(f"Parquet size: {sum(path.stat().st_size for path in parquet_files) / 1024 / 1024:,.2f} MB")

## Core coverage

In [ ]:
df.agg(
    F.count("*").alias("rows"),
    F.countDistinct("Ticker").alias("tickers"),
    F.min("Date").alias("min_date"),
    F.max("Date").alias("max_date"),
    F.min("Year").alias("min_year"),
    F.max("Year").alias("max_year"),
).show(truncate=False)

df.groupBy("Year").agg(
    F.count("*").alias("rows"),
    F.countDistinct("Ticker").alias("tickers"),
).orderBy("Year").show(100, truncate=False)

## Data quality checks

In [ ]:
required_columns = ["Ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
null_checks = [F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(f"{column}_nulls") for column in required_columns]
df.agg(*null_checks).show(truncate=False)

df.filter((F.col("Close") <= 0) | (F.col("Volume") < 0)).select(
    "Ticker", "Date", "Open", "High", "Low", "Close", "Volume"
).show(20, truncate=False)

## Price and volume distribution

In [ ]:
df.select("Open", "High", "Low", "Close", "Volume").describe().show(truncate=False)

df.approxQuantile("Close", [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99], 0.01), df.approxQuantile("Volume", [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99], 0.01)

## Top and bottom long-term movers

In [ ]:
from pyspark.sql import Window

first_window = Window.partitionBy("Ticker").orderBy(F.col("Date").asc())
last_window = Window.partitionBy("Ticker").orderBy(F.col("Date").desc())

first_rows = df.withColumn("rn", F.row_number().over(first_window)).filter("rn = 1").select(
    "Ticker", F.col("Date").alias("start_date"), F.col("Close").alias("start_close")
)
last_rows = df.withColumn("rn", F.row_number().over(last_window)).filter("rn = 1").select(
    "Ticker", F.col("Date").alias("end_date"), F.col("Close").alias("end_close")
)
movers = first_rows.join(last_rows, "Ticker").filter("start_close > 0 and end_close > 0").withColumn(
    "total_return_pct", (F.col("end_close") - F.col("start_close")) / F.col("start_close")
)

movers.orderBy(F.col("total_return_pct").desc()).show(20, truncate=False)
movers.orderBy(F.col("total_return_pct").asc()).show(20, truncate=False)

In [ ]:
df.unpersist()
spark.stop()